In [97]:
import os
import random
import re
import json
import math

import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch

In [99]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using {device}')

Using cpu


In [46]:
CLEANED_DATA_PATH = os.path.join("data", "cleaned_train_data.csv")
FINETUNE_MODEL_FOLDER = os.path.join("models", "mt5-large-finetune")

random.seed(42)
np.random.seed(42)

# Train Tokenizer

### Prepare data

We'll be training the tokenizer on Akkadian text only.

In [3]:
full_text_df = pd.read_csv(CLEANED_DATA_PATH, encoding = 'utf-8')

In [4]:
akkadian_corpus = full_text_df['transliteration_cleaned'].values
english_corpus = full_text_df['translation_cleaned'].dropna().values

print(f'There are {akkadian_corpus.shape[0]} pieces of akkadian text')
print(f'There are {english_corpus.shape[0]} pieces of english text')

There are 7953 pieces of akkadian text
There are 1561 pieces of english text


In [5]:
def get_tokenizer_train_corpus(full_corpus, batch_size = 1000):
    total_size = full_corpus.shape[0]
    return (
        full_corpus[i:i+batch_size].tolist() for i in range(0,total_size,batch_size)
    )

### Load pretrained tokenizer and train on new text

In [6]:
pretrained_tokenizer = AutoTokenizer.from_pretrained("models/mt5-large")

print(f'Loaded a tokenizer of type : {type(pretrained_tokenizer)}')

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
d:\conda_envs\deep-past\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
The tokenizer you are loading from 'mo

Loaded a tokenizer of type : <class 'transformers.models.t5.tokenization_t5_fast.T5TokenizerFast'>


In [7]:
incremental_tokenizer = pretrained_tokenizer.train_new_from_iterator(
    text_iterator = get_tokenizer_train_corpus(akkadian_corpus),
    vocab_size = 32000,
)

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Sanity check

In [8]:
akkadian_test = np.random.choice(akkadian_corpus, size = 3)
english_test = np.random.choice(english_corpus, size = 3)

In [9]:
for a_t in akkadian_test:
    print(f'Original: {a_t}')
    print(f'Pretrained tokenizer: {pretrained_tokenizer.tokenize(a_t)}')
    print(f'Incremental tokenizer: {incremental_tokenizer.tokenize(a_t)}')
    print("=="*10)

Original: <gap> išaqqal šumma lā išqul 1 ½ šiqlam <gap> ṣibtam Innā warḫem a- manā'em
Pretrained tokenizer: ['▁<', 'gap', '>', '▁iš', 'aqqa', 'l', '▁', 'šum', 'ma', '▁lā', '▁iš', 'qul', '▁1', '▁', '1⁄2', '▁ši', 'q', 'lam', '▁<', 'gap', '>', '▁', 'ṣi', 'b', 'tam', '▁Inn', 'ā', '▁war', 'ḫ', 'em', '▁', 'a', '-', '▁man', 'ā', "'", 'em']
Incremental tokenizer: ['▁<gap>', '▁išaqqal', '▁šumma', '▁lā', '▁išqul', '▁1', '▁1⁄2', '▁šiql', 'am', '▁<gap>', '▁ṣibt', 'am', '▁Innā', '▁warḫ', 'em', '▁a-', "▁manā'", 'em']
Original: 28 kutānē u 1 emārān ṣallāmum Šu-Bēlum ilqe šumma luqūtum anna ša awīlātem u Šu-Bēlem lā itu''ar ši'imšunu kasap Šu-Bēlum anna Šalim-Aššur išaqqal ina qerbem ⅔ manā 2 šiqlam kasap lu pazzurtam lu niplātem Šu-Bēlum išqul luqūtam ša Puzur-Aššur mer'ēn Išār-kit-Aššur ušēliāninni šēbum Kuzizīya šēbum Ennum-Aššur mer'ēn Babāya
Pretrained tokenizer: ['▁28', '▁ku', 'tā', 'nē', '▁', 'u', '▁1', '▁em', 'ārā', 'n', '▁ṣ', 'all', 'ām', 'um', '▁Šu', '-', 'B', 'ēl', 'um', '▁il', 'qe', '▁', '

In [10]:
for e_t in english_test:
    print(f'Original: {e_t}')
    print(f'Pretrained tokenizer: {pretrained_tokenizer.tokenize(e_t)}')
    print(f'Incremental tokenizer: {incremental_tokenizer.tokenize(e_t)}')
    print("=="*10)

Original: <big_gap>šia owes <big_gap> to Ali-ahum. He will pay at this coming harvest. Witnessed by Waldi-ilum.
Pretrained tokenizer: ['▁<', 'big', '_', 'gap', '>', 'šia', '▁', 'owe', 's', '▁<', 'big', '_', 'gap', '>', '▁to', '▁Ali', '-', 'a', 'hum', '.', '▁He', '▁will', '▁pay', '▁at', '▁this', '▁', 'coming', '▁', 'harvest', '.', '▁W', 'itness', 'ed', '▁by', '▁Wald', 'i', '-', 'i', 'lum', '.']
Incremental tokenizer: ['▁<big_gap>', 'ši', 'a', '▁', 'o', 'we', 's', '▁<big_gap>', '▁t', 'o', '▁Ali-', 'a', 'hu', 'm', '.', '▁', 'H', 'e', '▁', 'wi', 'l', 'l', '▁pa', 'y', '▁at', '▁t', 'h', 'is', '▁', 'c', 'o', 'm', 'i', 'n', 'g', '▁h', 'ar', 'v', 'es', 't', '.', '▁W', 'it', 'n', 'e', 'ss', 'ed', '▁b', 'y', '▁Wal', 'di', '-ilum', '.']
Original: 12 shekels of silver plus 8 ⅓ minas of good copper I deposited with Ennam-Aššur son of Amurrum-bāni after Nimar-Sīn <big_gap>
Pretrained tokenizer: ['▁12', '▁she', 'kel', 's', '▁of', '▁silver', '▁plus', '▁8', '▁1', '⁄', '3', '▁', 'minas', '▁of', '▁good', 

### Save new tokenizer

In [11]:
model = AutoModel.from_pretrained("models/mt5-large")
new_tokenizer = AutoTokenizer.from_pretrained("models/mt5-large")

The tokenizer you are loading from 'models/mt5-large' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Mark: the vocab size and embedding shape of mt5 model does not match. Check this: https://github.com/huggingface/transformers/issues/10528

In [12]:
# add new tokens
new_tokens = list(set(incremental_tokenizer.vocab.keys()) - set(pretrained_tokenizer.vocab.keys()))
num_added_tokens = new_tokenizer.add_tokens(new_tokens)
print(f'Added {num_added_tokens} new tokens')

# resize embedding
print(f'Previous embedding shape: {model.get_input_embeddings().weight.shape}')
model.resize_token_embeddings(len(new_tokenizer))
print(f'New embedding shape: {model.get_input_embeddings().weight.shape}')


Added 19054 new tokens
Previous embedding shape: torch.Size([250112, 1024])


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


New embedding shape: torch.Size([269154, 1024])


In [30]:
if not os.path.exists(FINETUNE_MODEL_FOLDER):
    os.mkdir(FINETUNE_MODEL_FOLDER)


new_tokenizer.save_pretrained(FINETUNE_MODEL_FOLDER)
model.save_pretrained(FINETUNE_MODEL_FOLDER)

# Task: Span Corruption (T5-style masked token modelling)

### Construct Training Data

In [89]:
# data filtering
MIN_SENT_LEN = 3  
MAX_SENT_LEN = 768
RESAMPLE_FACTOR = 0.3 # a text of T tokens will be resampled for T**RESAMPLE_FACTOR times
CORRUPTION_RATE = 0.15  # how much percent of tokens are corrupted
MAX_SPAN_LEN = 10  
SENTINEL_TOKENS = [f"<extra_id_{i}>" for i in range(100)]  

In [93]:
def filter_and_split_sentences(texts):
    """
    filter out short texts and split long texts
    """
    filtered_texts = []
    for text in texts:
        # NOTE: i don't use the trained tokenizer here 
        # because i don't want a single word being "half-masked"
        tokens = text.split() 

        # remove sentences that are too short
        if len(tokens) < MIN_SENT_LEN:
            continue
        # split sentences that are too long into equal-length chunks
        if len(tokens) > MAX_SENT_LEN:
            num_chunks = len(tokens) // MAX_SENT_LEN + 1
            for i in range(num_chunks):
                start_idx = i * MAX_SENT_LEN
                end_idx = start_idx + MAX_SENT_LEN
                chunk_tokens = tokens[start_idx:end_idx]
                if len(chunk_tokens) >= MIN_SENT_LEN:
                    filtered_texts.append(" ".join(chunk_tokens))
        else:
            filtered_texts.append(text.strip())
    return filtered_texts

def span_corrupt(text):
    """
    Performs span corruption on a piece of text
    returns the corrupted input and the label
    """
    tokens = text.split()
    num_tokens = len(tokens)
    
    num_corrupt_tokens = max(1, int(num_tokens * CORRUPTION_RATE)) 
    
    spans = []
    used_indices = set()  
    
    # each loop corrupts a span
    # loops until enough num of tokens are corrupted
    while sum([end - start + 1 for start, end in spans]) < num_corrupt_tokens:
        # randomly selects a span starting idx & a span length
        start = random.randint(0, num_tokens - 1)
        if start in used_indices:
            continue
        span_len = random.randint(1, MAX_SPAN_LEN)
        end = start + span_len - 1
        if end > num_tokens - 1:
            end = num_tokens - 1
            span_len = end - start + 1
        if any(not (end < s or start > e) for s, e in spans):
            continue
        
        spans.append((start, end))
        used_indices.update(range(start, end + 1))
    
    
    corrupted_tokens = tokens.copy()
    span_texts = []  
    spans_sorted = sorted(spans, key=lambda x: x[0])
    
    for idx, (start, end) in enumerate(spans_sorted):
        span_text = " ".join(tokens[start:end + 1])
        span_texts.append((SENTINEL_TOKENS[idx], span_text))
        corrupted_tokens[start:end + 1] = [SENTINEL_TOKENS[idx]]
    
    corrupted_input = " ".join(corrupted_tokens)
    target_span_text = " ".join([f"{sentinel} {text}" for sentinel, text in span_texts])
    
    return corrupted_input, target_span_text

In [94]:
filtered_akkadian_corpus = filter_and_split_sentences(akkadian_corpus)

masked_akkadian_corpus = []
for text in filtered_akkadian_corpus:
    resampled_times = math.ceil(len(text.split()) ** RESAMPLE_FACTOR)
    
    for _ in range(resampled_times):
        corrupted_input, target_span_text = span_corrupt(text)
        masked_akkadian_corpus.append(
            {
                "inputs": corrupted_input,
                "targets": target_span_text
            }
        )

In [96]:
print(len(masked_akkadian_corpus))

29626
